# Lab — Linear Solvers via a Black‑Box Heated Rod

You are given a **black box** that repeatedly produces linear systems of the form:

$$A x_{k+1} = b_k$$

- The matrix $A$ is **fixed**.
- The RHS $b_k$ changes at every step.
- Your task is to implement **three solvers** and use them to run an animation.

## Goal
A temperature sensor is placed at position $x_s$ on the rod. The simulation stops when the sensor first reaches a target temperature $T_{\text{target}}$.

You must report the **estimated time** $t^*$ when the threshold is reached.

## What you implement
- `gaussian_elimination(A, b) -> x`
- `jacobi(A, b, x0=None, tol=..., max_iter=...) -> (x, iters)`
- `gauss_seidel(A, b, x0=None, tol=..., max_iter=...) -> (x, iters)`



In [5]:
# Colab/Jupyter setup for inline animations
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rc

# Render FuncAnimation inline as JS/HTML
rc('animation', html='jshtml')

%matplotlib inline


## 1) Implement the solvers (placeholders)

Fill in these three functions. Keep the signatures.

**Iterative solvers must return `(x, iters)`**, where `iters` is the number of iterations used.


In [2]:

def gaussian_elimination(A, b):
    A = np.array(A, dtype=float, copy=True)
    b = np.array(b, dtype=float, copy=True).reshape(-1)

    n = A.shape[0]
    if A.shape != (n, n):
        raise ValueError("A must be square with shape (n, n).")
    if b.shape[0] != n:
        raise ValueError("b must have length n.")
    for i in range(n):
        pivot_row = i + np.argmax(np.abs(A[i:, i]))
        if np.isclose(A[pivot_row, i], 0.0):
            raise np.linalg.LinAlgError("Matrix is singular or nearly singular.")
        if pivot_row != i:
            A[[i, pivot_row]] = A[[pivot_row, i]]
            b[[i, pivot_row]] = b[[pivot_row, i]]
        for j in range(i + 1, n):
            factor = A[j, i] / A[i, i]
            A[j, i:] -= factor * A[i, i:]
            b[j] -= factor * b[i]
    x = np.zeros(n, dtype=float)
    for i in range(n - 1, -1, -1):
        if np.isclose(A[i, i], 0.0):
            raise np.linalg.LinAlgError("Matrix is singular or nearly singular.")
        x[i] = (b[i] - np.dot(A[i, i + 1:], x[i + 1:])) / A[i, i]

    return x

def jacobi(A, b, x0=None, tol=1e-8, max_iter=10000):
    A = np.asarray(A, dtype=float)
    b_in = np.asarray(b, dtype=float)
    b = b_in.ravel()
    n = A.shape[0]

    if x0 is None:
        x = np.zeros(n)
    else:
        x = np.asarray(x0, dtype=float).ravel().copy()

    D = np.diag(A)
    R = A - np.diag(D)
    b_norm = np.linalg.norm(b)
    if b_norm == 0.0:
        b_norm = 1.0

    iters = 0
    for k in range(1, max_iter + 1):
        x_new = (b - R @ x) / D
        iters = k
        if np.linalg.norm(A @ x_new - b) / b_norm <= tol:
            x = x_new
            break
        x = x_new
    if b_in.ndim == 2:
        x = x.reshape(-1, 1)
    return x, iters

def gauss_seidel(A, b, x0=None, tol=1e-8, max_iter=10000):
    A = np.asarray(A, dtype=float)
    b_in = np.asarray(b, dtype=float)
    b = b_in.ravel()
    n = A.shape[0]

    if x0 is None:
        x = np.zeros(n)
    else:
        x = np.asarray(x0, dtype=float).ravel().copy()

    b_norm = np.linalg.norm(b)
    if b_norm == 0.0:
        b_norm = 1.0

    iters = 0
    for k in range(1, max_iter + 1):
        for i in range(n):
            s1 = A[i, :i] @ x[:i]
            s2 = A[i, i+1:] @ x[i+1:]
            x[i] = (b[i] - s1 - s2) / A[i, i]
        iters = k
        if np.linalg.norm(A @ x - b) / b_norm <= tol:
            break

    if b_in.ndim == 2:
        x = x.reshape(-1, 1)
    return x, iters

## 3) Animation harness

This is the provided harness. It:
- builds one fixed matrix $A$ (hidden physics detail),
- repeatedly generates RHS vectors $b_k$ from the current state,
- solves $A x_{k+1} = b_k$ using your chosen solver,
- stops when the sensor reaches the target temperature.


In [3]:
from __future__ import annotations

import inspect
import time
from typing import Any, Callable, Dict, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML


# ----------------------------
# Adapter layer (tolerates minor signature / return differences)
# ----------------------------
def _normalize_x(x: Any, n: int) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    if x.ndim == 2 and x.shape[1] == 1:
        x = x[:, 0]
    x = x.reshape(-1)
    if x.shape[0] != n:
        raise ValueError(f"Solver returned shape {x.shape}, expected length {n}")
    return x


def _call_solver(
    fn: Callable[..., Any],
    A: np.ndarray,
    b: np.ndarray,
    *,
    x0: Optional[np.ndarray] = None,
    tol: float = 1e-6,
    max_iter: int = 1000,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    """Call student solver with mild signature/return robustness."""
    sig = inspect.signature(fn)
    params = sig.parameters

    kwargs: Dict[str, Any] = {}

    if x0 is not None:
        for name in ("x0", "x_init", "initial_guess"):
            if name in params:
                kwargs[name] = x0
                break

    if "tol" in params:
        kwargs["tol"] = tol
    elif "tolerance" in params:
        kwargs["tolerance"] = tol

    if "max_iter" in params:
        kwargs["max_iter"] = max_iter
    elif "max_iters" in params:
        kwargs["max_iters"] = max_iter
    elif "iterations" in params:
        kwargs["iterations"] = max_iter

    try:
        out = fn(A, b, **kwargs)
    except Exception:
        out = fn(A, b.reshape(-1, 1), **kwargs)

    info: Dict[str, Any] = {}
    if isinstance(out, tuple):
        x = out[0]
        if len(out) >= 2:
            info["iters"] = out[1]
        if len(out) >= 3:
            info["residual_reported"] = out[2]
    else:
        x = out

    return x, info


def rel_residual(A: np.ndarray, x: np.ndarray, b: np.ndarray) -> float:
    r = A @ x - b
    nb = np.linalg.norm(b)
    nr = np.linalg.norm(r)
    return float(nr / (nb + 1e-30))


# ----------------------------
# Black-box internals (not the point of the lab)
# ----------------------------
def build_A_dirichlet(n: int, alpha: float) -> np.ndarray:
    A = np.zeros((n, n), dtype=float)
    A[0, 0] = 1.0
    A[-1, -1] = 1.0
    for i in range(1, n - 1):
        A[i, i] = 1.0 + 2.0 * alpha
        A[i, i - 1] = -alpha
        A[i, i + 1] = -alpha
    return A


def heater_profile(x: np.ndarray, *, center: float, width: float, power: float) -> np.ndarray:
    return power * np.exp(-((x - center) / width) ** 2)


SOLVERS: Dict[str, Tuple[str, Callable[..., Any]]] = {
    "direct": ("Direct (Gaussian elimination)", gaussian_elimination),
    "jacobi": ("Iterative (Jacobi)", jacobi),
    "gs": ("Iterative (Gauss-Seidel)", gauss_seidel),
}


def run_animation(solver_key: str = "jacobi"):
    if solver_key not in SOLVERS:
        raise ValueError(f"solver_key must be one of {list(SOLVERS.keys())}")
    solver_name, solver_fn = SOLVERS[solver_key]

    # Parameters (tuned for a clear event)
    L = 1.0
    Nx = 70
    dx = L / (Nx - 1)

    dt = 0.5
    T_max = 9.0
    target_fps = 30

    D = 0.03
    k_cool = 0.01
    Tamb = 20.0
    T_target = 30.0

    sensor_x = 0.60 * L
    heater_x = 0.12 * L
    heater_w = 0.05 * L
    heater_power = 85.0

    tol = 1e-6
    max_iter = 1000

    xgrid = np.linspace(0.0, L, Nx)
    alpha = D * dt / (dx * dx)
    A = build_A_dirichlet(Nx, alpha)
    q = heater_profile(xgrid, center=heater_x, width=heater_w, power=heater_power)

    # state: v = T - Tamb
    v = np.zeros(Nx, dtype=float)

    sensor_idx = int(round(sensor_x / dx))
    sensor_idx = max(0, min(Nx - 1, sensor_idx))
    v_target = T_target - Tamb

    # Figure
    fig, (ax_rod, ax_line) = plt.subplots(
        2, 1,
        figsize=(9, 5.3),
        gridspec_kw={"height_ratios": [0.28, 0.72]},
        sharex=True,
    )

    rod_h = 10
    T_display = Tamb + v
    rod_img = np.repeat(T_display[None, :], rod_h, axis=0)
    im = ax_rod.imshow(
        rod_img,
        aspect="auto",
        extent=(0.0, L, 0.0, 1.0),
        origin="lower",
        interpolation="nearest",
        vmin=Tamb,
        vmax=max(T_target + 10.0, Tamb + 10.0),
    )
    ax_rod.set_yticks([])
    ax_rod.set_ylabel("rod")
    ax_rod.axvline(sensor_x, linestyle="--", linewidth=1)

    (line,) = ax_line.plot(xgrid, T_display, lw=2)
    ax_line.axvline(sensor_x, linestyle="--", linewidth=1)
    ax_line.axhline(T_target, linestyle="--", linewidth=1)

    ax_line.set_xlim(0.0, L)
    ax_line.set_ylim(Tamb - 5.0, max(T_target + 15.0, Tamb + 15.0))
    ax_line.set_xlabel("x")
    ax_line.set_ylabel("Temperature (°C)")

    title = ax_line.set_title("")
    textbox = ax_line.text(
        0.02, 0.98, "",
        transform=ax_line.transAxes,
        va="top",
        ha="left",
        fontsize=9,
        family="monospace",
    )

    # Stats + event detection
    t_sim = 0.0
    t0 = time.perf_counter()
    frame_count = 0

    last_ms = 0.0
    avg_ms = 0.0
    last_iters: Optional[Any] = None
    last_rr = 0.0

    event_reached = False
    event_time_est: Optional[float] = None
    prev_sensor_v = float(v[sensor_idx])
    prev_t = 0.0

    def update(_frame_idx: int):
        nonlocal v, t_sim, frame_count
        nonlocal last_ms, avg_ms, last_iters, last_rr
        nonlocal event_reached, event_time_est, prev_sensor_v, prev_t

        if event_reached:
            return (im, line, title, textbox)

        b = v + dt * (q - k_cool * v)
        b[0] = 0.0
        b[-1] = 0.0

        x0 = v.copy()

        start = time.perf_counter()
        v_raw, info = _call_solver(solver_fn, A, b, x0=x0, tol=tol, max_iter=max_iter)
        elapsed = time.perf_counter() - start

        v = _normalize_x(v_raw, Nx)
        v[0] = 0.0
        v[-1] = 0.0

        last_rr = rel_residual(A, v, b)
        last_iters = info.get("iters", None)

        frame_count += 1
        last_ms = elapsed * 1e3
        avg_ms += (last_ms - avg_ms) / frame_count

        prev_t = t_sim
        t_sim += dt

        sensor_v = float(v[sensor_idx])
        if sensor_v >= v_target:
            denom = sensor_v - prev_sensor_v
            frac = 1.0 if denom == 0.0 else (v_target - prev_sensor_v) / denom
            frac = min(1.0, max(0.0, frac))
            event_time_est = prev_t + dt * frac
            event_reached = True

        prev_sensor_v = sensor_v

        T_display = Tamb + v
        line.set_ydata(T_display)
        rod_img = np.repeat(T_display[None, :], rod_h, axis=0)
        im.set_data(rod_img)

        real_elapsed = time.perf_counter() - t0
        fps = frame_count / (real_elapsed + 1e-30)

        it_str = "-" if last_iters is None else str(last_iters)
        ev_str = "not reached"
        if event_reached and event_time_est is not None:
            ev_str = f"reached at t*={event_time_est:.3f}s"

        title.set_text(f"Heated rod with cooling | {solver_name}")
        textbox.set_text(
            "\n".join(
                [
                    f"sim t = {t_sim:7.2f} s   |   sensor x = {sensor_x:.3f}   |   target = {T_target:.1f}°C",
                    f"event: {ev_str}",
                    f"solve/frame: last = {last_ms:8.3f} ms   avg = {avg_ms:8.3f} ms   fps = {fps:5.1f}",
                    f"iters = {it_str:>8s}   |   relres = {last_rr:.2e}",
                ]
            )
        )

        if event_reached or t_sim >= T_max:
            try:
                ani.event_source.stop()  # type: ignore[name-defined]
            except Exception:
                pass

        return (im, line, title, textbox)

    frames = int(T_max / dt) + 1
    ani = FuncAnimation(fig, update, frames=frames, interval=1000 / target_fps, blit=False, repeat=False)

    plt.close(fig)  # prevent duplicate static figure output in notebooks
    return ani


## 4) Run the animation

Set `solver_key` to one of:
- `"direct"`
- `"jacobi"`
- `"gs"`

Then run the cell.


In [6]:
solver_key = "gs"  # change to: "direct" | "jacobi" | "gs"

ani = run_animation(solver_key)
ani


In [7]:
solver_key = "direct"  # change to: "direct" | "jacobi" | "gs"

ani = run_animation(solver_key)
ani


In [8]:
solver_key = "jacobi"  # change to: "direct" | "jacobi" | "gs"

ani = run_animation(solver_key)
ani
